# LCEL & Manual Tool Calling in LangChain

This notebook explains, section by section, what the companion script
[`lcel-and-manual-tool-calling.py`](lcel-and-manual-tool-calling.py) does and *why* it's written that way.
The `.py` file is the one meant to actually be run end-to-end; this notebook exists so you can
read and re-run each idea in isolation with the reasoning next to the code.

**Two ideas are covered:**
1. **LCEL (LangChain Expression Language)** - composing prompts/models/parsers with the `|` operator,
   plus the extra `Runnable` building blocks (`RunnableParallel`, `RunnablePassthrough`, `RunnableLambda`,
   `RunnableBranch`) that let you build non-linear pipelines without writing custom orchestration code.
2. **Manual tool calling** - instead of handing tool execution off to a prebuilt agent, you see exactly
   how a model's tool-call request gets read and executed, but expressed as an LCEL chain instead of a
   hand-written `while` loop.

## 1-2. Imports & Model Setup

Everything here composes `Runnable` objects, so the imports are mostly from `langchain_core.runnables`.
The model is Groq's `qwen/qwen3.8-27b` - the same model used elsewhere in this repo, because the
default `GROQ_MODEL` (`allam-2-7b`) doesn't support tool calling, and several sections below depend on it.

In [ ]:
import os
import sys
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    RunnableBranch,
)


load_dotenv()
sys.stdout.reconfigure(encoding="utf-8")

llm = ChatGroq(
    model="qwen/qwen3.8-27b",  # allam-2-7b (GROQ_MODEL) has no tool-calling support
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0,
)

## 3. LCEL Basics - the Pipe Operator

The `|` operator is LCEL's core idea: every stage (a prompt template, a chat model, an output parser)
implements the same `Runnable` interface, so `|` just feeds the output of the left side into the input
of the right side. `prompt | llm | StrOutputParser()` reads left-to-right exactly like a Unix pipe:
build the prompt -> send it to the model -> unwrap the plain string from the model's response object.

In [ ]:
# prompt | llm | parser: each stage is a Runnable, and | wires the output of
# one straight into the input of the next.
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this in exactly one sentence: {text}"
)
summarize_chain = summarize_prompt | llm | StrOutputParser()

print("=== Basic LCEL chain ===")
print(
    summarize_chain.invoke(
        {"text": "LCEL lets you compose prompts, models, and parsers with the | operator."}
    )
)

## 4. RunnableParallel - Fan Out to Multiple Chains

`RunnableParallel` runs several independent chains against the *same* input at the same time and
collects their outputs into one dict, keyed by whatever name you give each branch. Here the same
`{"text": ...}` input is summarized and translated in one `.invoke()` call instead of two.

In [ ]:
# Runs several chains on the same input at once and collects the results
# into a dict keyed by branch name.
translate_prompt = ChatPromptTemplate.from_template("Translate to French: {text}")
translate_chain = translate_prompt | llm | StrOutputParser()

parallel_chain = RunnableParallel(summary=summarize_chain, translation=translate_chain)

print("\n=== RunnableParallel ===")
print(parallel_chain.invoke({"text": "LangChain makes it easy to build LLM applications."}))

## 5. RunnablePassthrough - Carry the Original Input Forward

Once a chain transforms its input, the original value is normally gone. `RunnablePassthrough()`
is a no-op `Runnable` - it just returns whatever it's given - so wrapping it in a `RunnableParallel`
alongside a real chain lets you keep the original input *and* the transformed result side by side
(here: the question next to its answer).

In [ ]:
# Useful when a later step needs both the transformed value and the raw
# original input (e.g. showing the question next to its answer).
answer_prompt = ChatPromptTemplate.from_template("Answer briefly: {question}")
with_original_question = RunnableParallel(
    question=RunnablePassthrough(),
    answer=answer_prompt | llm | StrOutputParser(),
)

print("\n=== RunnablePassthrough ===")
print(with_original_question.invoke({"question": "What is LCEL short for?"}))

## 6. RunnableLambda - Plain Python as a Pipeline Step

Not every step needs to be an LLM call. `RunnableLambda` wraps an ordinary Python function so it can
sit inside a `|` chain like any other stage - useful for reshaping data, doing local computation, or
(as Section 9 below shows) running tool calls.

In [ ]:
# Wraps an ordinary function so it can sit in the | chain like any other
# Runnable - no special base class needed.
def word_count(text: str) -> dict:
    return {"text": text, "word_count": len(text.split())}


count_chain = RunnableLambda(word_count)

print("\n=== RunnableLambda ===")
print(count_chain.invoke("this sentence has six words total"))

## 7. RunnableBranch - Conditional Routing

`RunnableParallel` runs *every* branch. `RunnableBranch` runs *one* - it checks each
`(condition, runnable)` pair in order against the input and executes the first chain whose condition
is `True`, falling back to a default chain (the last, condition-less entry) if none match. Here, short
inputs get a one-line reply and everything else gets a detailed one.

In [ ]:
# Picks which chain to run based on a condition checked against the input,
# instead of every branch running like RunnableParallel does.
short_prompt = ChatPromptTemplate.from_template("Reply in one short sentence: {text}")
long_prompt = ChatPromptTemplate.from_template("Reply with a detailed paragraph: {text}")

routed_chain = RunnableBranch(
    (lambda x: len(x["text"]) < 30, short_prompt | llm | StrOutputParser()),
    long_prompt | llm | StrOutputParser(),  # default branch
)

print("\n=== RunnableBranch ===")
print(routed_chain.invoke({"text": "Explain LCEL."}))

## 8. Manual Tool Calling - Define Tools

Before tool calls can be executed manually, there need to be tools and a model that knows about them.
`llm.bind_tools([...])` attaches each tool's schema (name, description, argument types) to the model -
the model can now *request* a call, but nothing runs automatically; something still has to read
`ai_msg.tool_calls` and invoke the matching tool. That's what the next section does.

In [ ]:
@tool
def get_stock_price(ticker: str) -> str:
    """Look up the current price for a stock ticker symbol."""
    fake_prices = {"AAPL": 227.50, "GOOG": 178.20, "MSFT": 420.10}
    price = fake_prices.get(ticker.upper())
    return f"{ticker.upper()}: ${price}" if price else f"No price found for {ticker}"


@tool
def compare_prices(price_a: str, price_b: str) -> str:
    """Compare two price strings like 'AAPL: $227.50' and say which is higher."""
    val_a = float(price_a.split("$")[1])
    val_b = float(price_b.split("$")[1])
    return price_a if val_a > val_b else price_b


tools_by_name = {get_stock_price.name: get_stock_price, compare_prices.name: compare_prices}
llm_with_tools = llm.bind_tools([get_stock_price, compare_prices])

## 9. Manual Tool Calling, the LCEL Way

This is the section that ties both ideas together. Instead of a `while` loop that keeps calling the
model and checking `if ai_msg.tool_calls:` (the imperative style used in the
[`Fundamentals of building an AI agent`](../Fundamentals%20of%20building%20an%20AI%20agent) folder),
tool execution is wrapped in a `RunnableLambda` and piped directly after the tool-bound model:
`llm_with_tools | RunnableLambda(execute_tool_calls)`. Invoking that chain does one full round trip -
ask the model, run whatever tools it asked for - and returns the resulting messages, which are then
fed back into the model once more for the final answer. Same mechanics as a manual loop, just
expressed declaratively as a pipeline instead of an explicit loop.

In [ ]:
# Instead of a hand-rolled while loop, the tool-execution step is itself a
# RunnableLambda dropped into the pipe: llm_with_tools decides which tool(s)
# to call, execute_tool_calls runs them, and the results flow to the next
# invoke - the same round trip, expressed as a chain rather than a loop.
def execute_tool_calls(ai_msg):
    messages = [ai_msg]
    for call in ai_msg.tool_calls:
        result = tools_by_name[call["name"]].invoke(call["args"])
        messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    return messages


tool_call_chain = llm_with_tools | RunnableLambda(execute_tool_calls)

print("\n=== Manual tool calling via LCEL ===")
question = [HumanMessage("What's the price of AAPL?")]
tool_result_messages = tool_call_chain.invoke(question)
final_answer = llm_with_tools.invoke(question + tool_result_messages)
print(final_answer.content)

## 10. Batch and Stream - Runnable Methods Beyond invoke()

Because every LCEL chain implements the same `Runnable` interface, `.batch()` and `.stream()` work on
*any* chain built with `|`, with no extra code: `.batch()` runs the chain over a list of inputs, and
`.stream()` yields output incrementally as the model generates it instead of waiting for the full response.

In [ ]:
# Every LCEL chain gets .batch() and .stream() for free, with no extra code.
print("\n=== .batch() - run the same chain over multiple inputs at once ===")
batch_results = summarize_chain.batch(
    [{"text": "LCEL supports batching."}, {"text": "LCEL supports streaming."}]
)
for r in batch_results:
    print("-", r)

print("\n=== .stream() - receive the response as it's generated ===")
for chunk in summarize_chain.stream({"text": "Streaming shows tokens as they arrive."}):
    print(chunk, end="", flush=True)
print()

## Key Takeaways

- LCEL's `|` operator composes any `Runnable`s (prompts, models, parsers, plain functions) into a pipeline.
- `RunnableParallel` fans out, `RunnableBranch` routes, `RunnableLambda` drops in plain Python,
  `RunnablePassthrough` preserves the original input alongside derived results.
- Manual tool calling means *you* read `ai_msg.tool_calls` and invoke the matching tool - nothing happens
  automatically just from calling `bind_tools()`. Expressing that execution step as a `RunnableLambda`
  inside a `|` chain is the LCEL-idiomatic alternative to a hand-rolled loop.
- Every LCEL chain gets `.invoke()`, `.batch()`, and `.stream()` for free.